# TICODER Live Demo: Human PASS/FAIL Feedback

Use this notebook after your slides. The notebook is intentionally short: first a tiny presentation recap, then the live demo.

## Presentation Recap

**Problem:** the LLM gives many possible code solutions, but we must choose one.

**TICODER idea:** generate code candidates and test candidates, then ask the user about the most useful test.

**This demo difference:** the original experiment uses an oracle/reference implementation. Here, the audience types `PASS`, `FAIL`, or `UNDEFINED`.

**Our extension:** soft scoring keeps all candidates alive and uses entropy to decide when to stop early.

## Demo Setup

This notebook always uses the Anthropic API for generation. Paste the API key in the variable below before running in Colab.

I did not store the real key in the file because notebooks are easy to share by accident.

In [ ]:
# Paste the real key here in Colab before running.
ANTHROPIC_API_KEY = "PASTE_YOUR_ANTHROPIC_KEY_HERE"

MODEL = "claude-haiku-4-5"
MAX_TOKENS = 150
TEMPERATURE = 0.8

# Small numbers keep the live demo fast and cheap.
N_CODE_CANDIDATES = 6
N_TEST_CANDIDATES = 5
MAX_INTERACTIONS = 5
ENTROPY_THRESHOLD = 0.5

if not ANTHROPIC_API_KEY.startswith("sk-ant-"):
    raise ValueError("Paste your Anthropic API key into ANTHROPIC_API_KEY before running the demo.")

In [ ]:
import ast
import html
import math
import re
import subprocess
import sys
import textwrap
from IPython.display import HTML, Markdown, display

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import anthropic
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "anthropic"])
    import anthropic

PASS = "PASS"
FAIL = "FAIL"
UNDEFINED = "UNDEFINED"

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def section(title, body=""):
    display(HTML(f"""
    <div style='border-left:6px solid #2563eb; padding:10px 14px; margin:12px 0; background:#f8fbff'>
      <div style='font-size:21px; font-weight:700; color:#111827'>{html.escape(title)}</div>
      <div style='font-size:15px; color:#374151; margin-top:3px'>{html.escape(body)}</div>
    </div>
    """))

def show_table(rows):
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        for row in rows:
            print(row)

def call_claude(user_prompt, system_prompt):
    msg = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}],
    )
    return msg.content[0].text.strip()

def extract_backtick_content(text):
    match = re.search(r"```(?:python)?\s*(.*?)```", text, re.S)
    if match:
        return match.group(1).strip()
    match = re.search(r"`([^`]+)`", text, re.S)
    if match:
        return match.group(1).strip()
    return text.strip()

def short_code(code, width=90):
    line = " ".join(code.strip().split())
    return line[:width] + ("..." if len(line) > width else "")

## Step 1: Choose the Task

This fixed task keeps the demo understandable while the candidate codes and tests are generated live.

In [ ]:
TASK = {
    "name": "remove_duplicates",
    "signature": "def remove_duplicates(nums):",
    "description": "Return a list with duplicate values removed from nums.",
}

DEMO_INTENT = "For this demo, answer PASS only if the result keeps the first occurrence order."

section("Task", TASK["description"])
show_table([
    {"Field": "Function signature", "Value": TASK["signature"]},
    {"Field": "Audience job", "Value": "Judge generated tests as PASS, FAIL, or UNDEFINED"},
    {"Field": "Demo intent", "Value": DEMO_INTENT},
])

## Step 2: Generate Code and Tests with the API

In [ ]:
CODE_SYSTEM = (
    "Suppose you are a code completion engine. Complete the given Python function. "
    "Return only the completed function, surrounded by backticks."
)

TEST_SYSTEM = (
    "Suppose you are a code completion engine. Generate one Python assert statement "
    "for the given function. Return only one assertion, surrounded by backticks."
)

def build_code_prompt():
    return f"""Complete the following Python function:

{TASK['signature']}

The context of the function is:
{TASK['description']}

Surround the function with ` and ` tags.
Do not explain the function, just complete the function.
"""

def build_test_prompt():
    return f"""Context of the function is:

{TASK['description']}

The function is defined as follows:
{TASK['signature']}
    pass

Generate exactly one Python assert statement for this function.
The assertion must call {TASK['name']}.
Surround the assertion with ` and ` tags.
Do not explain it.
"""

def prune_equivalent_items(items):
    unique_items = []
    for item in items:
        if item not in unique_items:
            unique_items.append(item)
    return unique_items

DEMO_MIN_UNIQUE_CODES = 6
DEMO_MIN_UNIQUE_TESTS = 5

def build_demo_fallback_code_prompt(existing_items):
    existing_preview = "\n\n".join(
        f"Existing candidate {i+1}:\n{code}"
        for i, code in enumerate(existing_items)
    )
    return f"""Complete the following Python function:

{TASK['signature']}

The context of the function is:
{TASK['description']}

For a live TICODER demo, generate a plausible alternative candidate that is meaningfully different from the existing candidates below.
It may be correct or subtly imperfect, but it must be valid Python and must keep the same function signature.

Existing candidates:
{existing_preview}

Surround the function with ` and ` tags.
Do not explain the function, just complete the function.
"""

def generate_demo_fallback_codes(unique_items, min_unique, max_extra_attempts=30):
    for attempt in range(max_extra_attempts):
        if len(unique_items) >= min_unique:
            break
        raw = call_claude(build_demo_fallback_code_prompt(unique_items), CODE_SYSTEM)
        item = extract_backtick_content(raw)
        valid = item.strip().startswith("def ")
        if valid and item not in unique_items:
            unique_items.append(item)
    return unique_items

def build_demo_fallback_test_prompt(existing_items):
    existing_preview = "\n".join(
        f"Existing test {i+1}: {test}"
        for i, test in enumerate(existing_items)
    )
    return f"""Context of the function is:

{TASK['description']}

The function is defined as follows:
{TASK['signature']}
    pass

For a live TICODER demo, generate one plausible alternative assert statement that is meaningfully different from the existing tests below.
Use different inputs, edge cases, ordering, duplicates, empty input, or boundary cases when possible.

Existing tests:
{existing_preview}

The assertion must call {TASK['name']}.
Surround the assertion with ` and ` tags.
Do not explain it.
"""

def generate_demo_fallback_tests(unique_items, min_unique, max_extra_attempts=30):
    for attempt in range(max_extra_attempts):
        if len(unique_items) >= min_unique:
            break
        raw = call_claude(build_demo_fallback_test_prompt(unique_items), TEST_SYSTEM)
        item = extract_backtick_content(raw).strip()
        item = item.splitlines()[0].strip() if item else ""
        valid = item.startswith("assert ") and TASK["name"] in item
        if valid and item not in unique_items:
            unique_items.append(item)
    return unique_items

def generate_and_prune_items(kind, target_n, max_attempts=None):
    if max_attempts is None:
        max_attempts = target_n
    items = []
    for attempt in range(max_attempts):
        if len(items) >= target_n:
            break
        if kind == "code":
            raw = call_claude(build_code_prompt(), CODE_SYSTEM)
            item = extract_backtick_content(raw)
            valid = item.strip().startswith("def ")
        else:
            raw = call_claude(build_test_prompt(), TEST_SYSTEM)
            item = extract_backtick_content(raw).strip()
            item = item.splitlines()[0].strip() if item else ""
            valid = item.startswith("assert ") and TASK["name"] in item
        if valid:
            items.append(item)
    if len(items) < target_n:
        raise RuntimeError(
            f"Only generated {len(items)} valid {kind} item(s). "
            f"Rerun this cell or increase max_attempts."
        )
    unique_items = prune_equivalent_items(items)
    if kind == "code" and len(unique_items) < min(DEMO_MIN_UNIQUE_CODES, target_n):
        unique_items = generate_demo_fallback_codes(
            unique_items,
            min(DEMO_MIN_UNIQUE_CODES, target_n),
        )
    if kind == "test" and len(unique_items) < min(DEMO_MIN_UNIQUE_TESTS, target_n):
        unique_items = generate_demo_fallback_tests(
            unique_items,
            min(DEMO_MIN_UNIQUE_TESTS, target_n),
        )
    return unique_items

section("Calling Anthropic API", "Generating live code candidates and test candidates now.")
CODE_CANDIDATES = generate_and_prune_items("code", N_CODE_CANDIDATES)
TEST_CANDIDATES = generate_and_prune_items("test", N_TEST_CANDIDATES)

section("Generated code candidates", "These are the candidates TICODER will rank.")
for i, code in enumerate(CODE_CANDIDATES):
    display(Markdown(f"### C{i}"))
    display(Markdown(f"```python\n{code}\n```"))

section("Generated tests", "These are candidate questions for the audience.")
for i, test in enumerate(TEST_CANDIDATES, 1):
    display(Markdown(f"**T{i}:** `{test}`"))

## Step 3: Build the Execution Matrix

Each code candidate is executed against each generated test. The result becomes `pass`, `fail`, or `crash`.

In [ ]:
def run_assertion_on_code(code, assertion):
    namespace = {}
    try:
        exec(code, namespace)
        exec(assertion, namespace)
        return "pass"
    except AssertionError:
        return "fail"
    except Exception:
        return "crash"

def build_execution_matrix(codes, tests):
    matrix = {}
    for test in tests:
        matrix[test] = {}
        for idx, code in enumerate(codes):
            matrix[test][idx] = run_assertion_on_code(code, test)
    return matrix

MATRIX = build_execution_matrix(CODE_CANDIDATES, TEST_CANDIDATES)

rows = []
for t_idx, test in enumerate(TEST_CANDIDATES, 1):
    row = {"Test": f"T{t_idx}", "Assertion": test}
    for c_idx in range(len(CODE_CANDIDATES)):
        row[f"C{c_idx}"] = MATRIX[test][c_idx]
    rows.append(row)

section("Execution matrix", "This table shows how candidates split on each generated test.")
show_table(rows)

## Step 4: Select the Best Test to Ask

A good test divides candidates into two behavior clusters: `PASS` and `FAIL`.

`s_discr = min(PASS cluster, FAIL cluster) / max(PASS cluster, FAIL cluster)`

In [ ]:
def pass_fail_counts(test, active_indices):
    n_pass = sum(1 for i in active_indices if MATRIX[test][i] == "pass")
    n_fail = sum(1 for i in active_indices if MATRIX[test][i] == "fail")
    n_crash = sum(1 for i in active_indices if MATRIX[test][i] == "crash")
    return n_pass, n_fail, n_crash

def s_discr(test, active_indices):
    n_pass, n_fail, _ = pass_fail_counts(test, active_indices)
    top = max(n_pass, n_fail)
    bottom = min(n_pass, n_fail)
    return 0.0 if top == 0 else bottom / top

def shannon_entropy(test, active_indices):
    n_pass, n_fail, _ = pass_fail_counts(test, active_indices)
    total = n_pass + n_fail
    if total == 0:
        return 0.0
    H = 0.0
    for count in (n_pass, n_fail):
        if count > 0:
            p_i = count / total
            H -= p_i * math.log2(p_i)
    return H

def rank_tests(available_tests, active_indices):
    rows = []
    for test in available_tests:
        n_pass, n_fail, n_crash = pass_fail_counts(test, active_indices)
        rows.append({
            "Test ID": f"T{TEST_CANDIDATES.index(test)+1}",
            "Test": test,
            "PASS cluster": n_pass,
            "FAIL cluster": n_fail,
            "CRASH": n_crash,
            "s_discr": round(s_discr(test, active_indices), 3),
            "H": round(shannon_entropy(test, active_indices), 3),
            "_test": test,
        })
    rows.sort(key=lambda r: (r["s_discr"], r["H"]), reverse=True)
    return rows

ACTIVE = list(range(len(CODE_CANDIDATES)))
AVAILABLE_TESTS = TEST_CANDIDATES.copy()
SOFT_SCORES = {i: 0 for i in ACTIVE}
HISTORY = []

section("Test ranking", "The first row is the test we ask about first.")
show_table([{k: v for k, v in r.items() if not k.startswith("_")} for r in rank_tests(AVAILABLE_TESTS, ACTIVE)])

## Step 5: Ask the Audience

Run the next cell once per interaction. Type the audience answer: `PASS`, `FAIL`, or `UNDEFINED`.

In [ ]:
def score_delta(execution_result, human_answer):
    if human_answer == UNDEFINED or execution_result == "crash":
        return 0
    if human_answer == PASS:
        return 1 if execution_result == "pass" else -1
    if human_answer == FAIL:
        return 1 if execution_result == "fail" else -1
    return 0

def ranked_codes():
    rows = []
    for idx, code in enumerate(CODE_CANDIDATES):
        rows.append({"Candidate": f"C{idx}", "Soft score": SOFT_SCORES[idx], "Preview": short_code(code)})
    rows.sort(key=lambda r: (-r["Soft score"], int(r["Candidate"][1:])))
    return rows

def hard_pruning_survivors(test, human_answer):
    if human_answer == UNDEFINED:
        return ACTIVE.copy()
    survivors = []
    for idx in ACTIVE:
        behavior = MATRIX[test][idx]
        if human_answer == PASS and behavior != "fail":
            survivors.append(idx)
        elif human_answer == FAIL and behavior != "pass":
            survivors.append(idx)
    return survivors if survivors else ACTIVE.copy()

def run_interaction():
    global AVAILABLE_TESTS
    m = len(HISTORY) + 1
    if m > MAX_INTERACTIONS or not AVAILABLE_TESTS:
        section("Stop", "No more interactions are available.")
        return

    ranked = rank_tests(AVAILABLE_TESTS, ACTIVE)
    chosen = ranked[0]["_test"]
    H = shannon_entropy(chosen, ACTIVE)

    section(f"Interaction {m}", "Ask the highest-ranked test.")
    show_table([{k: v for k, v in ranked[0].items() if not k.startswith("_")}])
    print("\nAudience question:")
    print(chosen)

    while True:
        human_answer = input("Type PASS, FAIL, or UNDEFINED: ").strip().upper()
        if human_answer in {PASS, FAIL, UNDEFINED}:
            break
        print("Please type PASS, FAIL, or UNDEFINED.")

    update_rows = []
    for idx in ACTIVE:
        behavior = MATRIX[chosen][idx]
        delta = score_delta(behavior, human_answer)
        old = SOFT_SCORES[idx]
        SOFT_SCORES[idx] += delta
        update_rows.append({
            "Candidate": f"C{idx}",
            "Behavior": behavior,
            "Human answer": human_answer,
            "Delta": delta,
            "Old": old,
            "New": SOFT_SCORES[idx],
        })

    hard_keep = hard_pruning_survivors(chosen, human_answer)
    HISTORY.append({"m": m, "test": chosen, "answer": human_answer, "H": H})
    AVAILABLE_TESTS = [t for t in AVAILABLE_TESTS if t != chosen]

    section("Soft scoring update", "Agree = +1, disagree = -1, crash/undefined = 0.")
    show_table(update_rows)

    section("Current best candidates", "Soft scoring ranks by accumulated evidence.")
    show_table(ranked_codes())

    decision = "Stop early" if H < ENTROPY_THRESHOLD else "Ask next test"
    section("Entropy decision", f"H = {H:.3f}. Decision: {decision}.")
    print("Hard pruning would keep:", ", ".join(f"C{i}" for i in hard_keep))

run_interaction()

In [ ]:
# Run again for the next audience question.
run_interaction()

## Final Summary

In [ ]:
section("Final result", "The top row is the selected implementation.")
show_table(ranked_codes())

section("Interaction history", "What the audience answered and what entropy saw.")
show_table([
    {"m": h["m"], "Answer": h["answer"], "H": round(h["H"], 3), "Test": h["test"]}
    for h in HISTORY
])